In [13]:
#FINAL model

from src.models.model_saving import load_model_from_s3
from src.models.model_baseline_majority_vote import SoftVoteEnsemble
from src.models.train_catboost import CatBoostAutoCat

# 1 — Load ensemble model and fix catboost args
ensemble_model = load_model_from_s3(bucket=S3_BUCKET_FINAL_MODELS, model_name="ensemble_model")
# Catboost internal arg
cat_model    = ensemble_model.models[ensemble_model.names.index("CAT")]
cat_features = X_test_noenc_nonorm.select_dtypes(include=["object", "category", "string"]).columns.tolist()
cat_model._cat_features_fitted = cat_features

# 2 — Re-attach client_code to test datasets (row order preserved)
X_test_enc_nonorm["client_code"]   = client_codes
X_test_noenc_nonorm["client_code"] = client_codes
X_test_enc_norm["client_code"]     = client_codes


# 3 — Define test ensemble dataset
base = lambda df: df.drop(columns=[Config.KEY_COLUMN], errors="ignore")
X_test_ensemble = {
    "XGB": base(X_test_enc_nonorm)[features_xgb],
    "CAT": base(X_test_noenc_nonorm)[features_cat],
    "LR":  base(X_test_enc_norm)[features_lr],
}

# 4 — Define clip client lists
nps_clients     = X_test_noenc_nonorm[X_test_noenc_nonorm["client_nps_churn_mention_n"] > 0]["client_code"].tolist()
mail_clients    = X_test_noenc_nonorm[X_test_noenc_nonorm["interaction_mail_churn_mention"] > 0]["client_code"].tolist()
engaged_clients = X_test_noenc_nonorm[X_test_noenc_nonorm["client_toujours_engage"] == 1]["client_code"].tolist()

# 5 — Build final model with clip rules
final_model = (
    FinalModel(ensemble_model, global_threshold=Config.GLOBAL_THRESHOLD)
    .add_clip(nps_clients,     proba=0.95, label="NPS clip     (→0.95):")
    .add_clip(mail_clients,    proba=0.95, label="mail clip    (→0.95):")
    .add_clip(engaged_clients, proba=0.0,  label="engaged clip  (→0.0):")
)


ModuleNotFoundError: No module named 'src.models.model_baseline_majority_vote'

In [72]:
y_true = df[Config.TARGET_COLUMN]

In [80]:
client_codes.shape

(38449,)

In [83]:
probas=final_model.predict_proba(X_test_ensemble, client_codes=client_codes)

In [89]:
# 8 — Extract & analyze clip impact
# step 6
results = final_model.predict_final(X_test_ensemble, client_codes=client_codes)
# Merge y_true into results
results["y_true"] = y_true

# Clipped clients only
clipped_mask = results["churn_proba"] != results["churn_proba_clipped"]
clipped      = results[clipped_mask].copy()

# AUC comparison
from sklearn.metrics import roc_auc_score

auc_raw     = roc_auc_score(results["y_true"], results["churn_proba"])
auc_clipped = roc_auc_score(results["y_true"], results["churn_proba_clipped"])

print(f"AUC raw:     {auc_raw:.4f}")
print(f"AUC clipped: {auc_clipped:.4f}  (delta: {auc_clipped - auc_raw:+.4f})")

# Clip impact per rule
print("\n── Clip analysis ──────────────────────────────")
for clients, proba_value, label in final_model.clip_rules:
    mask   = clipped["client_code"].isin(clients)
    subset = clipped[mask]
    if subset.empty:
        continue
    print(f"\n{label}")
    print(f"  affected:      {len(subset)} clients")
    print(f"  true churn rate:  {subset['y_true'].mean()*100:.1f}%")   # is the clip direction correct?
    print(f"  avg raw proba:    {subset['churn_proba'].mean():.3f}")
    print(f"  forced to:        {proba_value}")

[0] Initial churn rate:        20.6%  (7906 clients)
[1] NPS clip     (→0.95):          proba forced to 0.95  (20 clients affected)
[2] mail clip    (→0.95):          proba forced to 0.95  (1786 clients affected)
[3] engaged clip  (→0.0):          proba forced to 0.0  (3047 clients affected)
AUC raw:     0.7892
AUC clipped: 0.7779  (delta: -0.0113)

── Clip analysis ──────────────────────────────

NPS clip     (→0.95):
  affected:      20 clients
  true churn rate:  35.0%
  avg raw proba:    0.482
  forced to:        0.95

mail clip    (→0.95):
  affected:      1786 clients
  true churn rate:  16.9%
  avg raw proba:    0.385
  forced to:        0.95

engaged clip  (→0.0):
  affected:      3047 clients
  true churn rate:  0.5%
  avg raw proba:    0.041
  forced to:        0.0


In [16]:
import Config
from sklearn.metrics import roc_auc_score
from src.models.model_saving import load_model_from_s3
from src.data_processing.data_load import load_data_processed_from_S3, data_loading
from src.models.model_baseline import SoftVoteEnsemble, FinalModel

# ── 1. Load ensemble model ────────────────────────────────────────────────────
ensemble_model = load_model_from_s3(bucket=Config.S3_BUCKET_FINAL_MODELS, model_name="ensemble_model")

xgb_model = ensemble_model.models[ensemble_model.names.index("XGB")]
lr_model  = ensemble_model.models[ensemble_model.names.index("LR")]
cat_model = ensemble_model.models[ensemble_model.names.index("CAT")]

features_xgb = xgb_model.get_booster().feature_names
features_cat = cat_model.feature_names_
features_lr  = lr_model.feature_names_in_

base = lambda df: df.drop(columns=[Config.KEY_COLUMN], errors="ignore")

# ── 2. TEST (validation split) ────────────────────────────────────────────────
X_test_enc_norm     = load_data_processed_from_S3("X_test_enc_norm")
X_test_enc_nonorm   = load_data_processed_from_S3("X_test_enc_nonorm")
X_test_noenc_nonorm = load_data_processed_from_S3("X_test_noenc_nonorm")
y_test              = load_data_processed_from_S3("y_test_enc_norm").values  # same y regardless of encoding

df_test, _, _, _, _ = data_loading("validation")
client_codes_test   = df_test[Config.KEY_COLUMN].values

X_test_enc_norm["client_code"]     = client_codes_test
X_test_enc_nonorm["client_code"]   = client_codes_test
X_test_noenc_nonorm["client_code"] = client_codes_test

cat_model._cat_features_fitted = X_test_noenc_nonorm.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

X_test_ensemble = {
    "XGB": base(X_test_enc_nonorm)[features_xgb],
    "CAT": base(X_test_noenc_nonorm)[features_cat],
    "LR":  base(X_test_enc_norm)[features_lr],
}

nps_clients     = X_test_noenc_nonorm[X_test_noenc_nonorm["client_nps_churn_mention_n"] > 0]["client_code"].tolist()
mail_clients    = X_test_noenc_nonorm[X_test_noenc_nonorm["interaction_mail_churn_mention"] > 0]["client_code"].tolist()
engaged_clients = X_test_noenc_nonorm[X_test_noenc_nonorm["client_toujours_engage"] == 1]["client_code"].tolist()

final_model = (
    FinalModel(ensemble_model, global_threshold=Config.GLOBAL_THRESHOLD)
    .add_clip(nps_clients,     proba=0.55, label="NPS clip:",     only_if_below=0.3)
    .add_clip(mail_clients,    proba=0.55, label="mail clip:",    only_if_below=0.3)
    .add_clip(engaged_clients, proba=0.0,  label="engaged clip:")
)

print("\n══ TEST RESULTS ═════════════════════════════════════════")
results_test        = final_model.predict_final(X_test_ensemble, client_codes=client_codes_test)
results_test["y_true"] = y_test

auc_raw_test     = roc_auc_score(results_test["y_true"], results_test["churn_proba"])
auc_clipped_test = roc_auc_score(results_test["y_true"], results_test["churn_proba_clipped"])
print(f"AUC raw:     {auc_raw_test:.4f}")
print(f"AUC clipped: {auc_clipped_test:.4f}  (delta: {auc_clipped_test - auc_raw_test:+.4f})")

# ── 3. EVAL (no labels) ───────────────────────────────────────────────────────
X_eval_enc_norm     = load_data_processed_from_S3("X_eval_enc_norm")
X_eval_enc_nonorm   = load_data_processed_from_S3("X_eval_enc_nonorm")
X_eval_noenc_nonorm = load_data_processed_from_S3("X_eval_noenc_nonorm")

client_codes_eval   = X_eval_noenc_nonorm[Config.KEY_COLUMN].values

X_eval_enc_norm["client_code"]     = client_codes_eval
X_eval_enc_nonorm["client_code"]   = client_codes_eval
X_eval_noenc_nonorm["client_code"] = client_codes_eval

cat_model._cat_features_fitted = X_eval_noenc_nonorm.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

X_eval_ensemble = {
    "XGB": base(X_eval_enc_nonorm)[features_xgb],
    "CAT": base(X_eval_noenc_nonorm)[features_cat],
    "LR":  base(X_eval_enc_norm)[features_lr],
}

nps_clients_eval     = X_eval_noenc_nonorm[X_eval_noenc_nonorm["client_nps_churn_mention_n"] > 0]["client_code"].tolist()
mail_clients_eval    = X_eval_noenc_nonorm[X_eval_noenc_nonorm["interaction_mail_churn_mention"] > 0]["client_code"].tolist()
engaged_clients_eval = X_eval_noenc_nonorm[X_eval_noenc_nonorm["client_toujours_engage"] == 1]["client_code"].tolist()

final_model_eval = (
    FinalModel(ensemble_model, global_threshold=Config.GLOBAL_THRESHOLD)
    .add_clip(nps_clients_eval,     proba=0.55, label="NPS clip:",     only_if_below=0.3)
    .add_clip(mail_clients_eval,    proba=0.55, label="mail clip:",    only_if_below=0.3)
    .add_clip(engaged_clients_eval, proba=0.0,  label="engaged clip:")
)

print("\n══ EVAL RESULTS ═════════════════════════════════════════")
results_eval = final_model_eval.predict_final(X_eval_ensemble, client_codes=client_codes_eval)

print(f"Churn rate (clipped): {results_eval['churn_pred'].mean()*100:.1f}%  ({results_eval['churn_pred'].sum()} clients)")
print(f"Avg proba raw:        {results_eval['churn_proba'].mean():.4f}")
print(f"Avg proba clipped:    {results_eval['churn_proba_clipped'].mean():.4f}")


══ TEST RESULTS ═════════════════════════════════════════
[0] Initial churn rate:  19.2%  (7372 clients)
[1] NPS clip:                      → 0.55 (only if below 0.3)  (3 clients affected)
[2] mail clip:                     → 0.55 (only if below 0.3)  (654 clients affected)
[3] engaged clip:                  → 0.0 (unconditional)  (3047 clients affected)
AUC raw:     0.7892
AUC clipped: 0.7868  (delta: -0.0024)

══ EVAL RESULTS ═════════════════════════════════════════
[0] Initial churn rate:  19.3%  (17306 clients)
[1] NPS clip:                      → 0.55 (only if below 0.3)  (5 clients affected)
[2] mail clip:                     → 0.55 (only if below 0.3)  (1463 clients affected)
[3] engaged clip:                  → 0.0 (unconditional)  (7173 clients affected)
Churn rate (clipped): 19.3%  (17306 clients)
Avg proba raw:        0.2995
Avg proba clipped:    0.2999


In [84]:
roc_auc_score(y_true, probas)

0.777871955921393

In [86]:
final_model.predict_final(X_test_ensemble, client_codes=client_codes)

[0] Initial churn rate:        20.6%  (7906 clients)
[1] NPS clip     (→0.95):          proba forced to 0.95  (20 clients affected)
[2] mail clip    (→0.95):          proba forced to 0.95  (1786 clients affected)
[3] engaged clip  (→0.0):          proba forced to 0.0  (3047 clients affected)


,client_code,churn_proba,churn_proba_clipped,churn_pred
0,22604,0.078517,0.078517,0
1,23318,0.096164,0.096164,0
2,24809,0.184057,0.184057,0
3,25201,0.070334,0.070334,0
4,25320,0.048844,0.048844,0
...,...,...,...,...
38444,16813630,0.018975,0.000000,0
38445,16814134,0.016366,0.000000,0
38446,16814470,0.011965,0.000000,0
38447,16816206,0.021863,0.000000,0


In [ ]:

# 6 — Run & extract results
results = final_model.predict_final(X_test_ensemble, client_codes=client_codes)

# 7 — Extract probas and predictions
churn_proba = results["churn_proba"]  # raw soft-vote probabilities
churn_pred  = results["churn_pred"]   # threshold only, no clips
churn_final = results["churn_final"]  # threshold + all clips applied

churn_proba

In [ ]:
# 6 — Run & extract results
results = final_model.predict_final(X_test_ensemble, client_codes=client_codes)

# 7 — Extract probas and predictions
churn_proba = results["churn_proba"]  # raw soft-vote probabilities
churn_pred  = results["churn_pred"]   # threshold only, no clips
churn_final = results["churn_final"]  # threshold + all clips applied

In [ ]:
#FINAL model

from src.models.model_saving import load_model_from_s3

# 1 — Load ensemble model
ensemble_model = load_model_from_s3(bucket=S3_BUCKET_FINAL_MODELS, model_name="ensemble_model")

# 2 — Extract client codes (only available in noenc_nonorm)
client_codes = load_data_processed_from_S3("X_eval_noenc_nonorm")[Config.KEY_COLUMN].values

# 3 — Define test ensemble dataset

X_test_ensemble = {
    "XGB": base(X_test_enc_nonorm)[features_xgb],
    "CAT": base(X_test_noenc_nonorm)[features_cat],
    "LR":  base(X_test_enc_norm)[features_lr],
}

# 4 — Define clip client lists
nps_clients     = X_test_noenc_nonorm[X_test_noenc_nonorm["client_nps_churn_mention_n"] > 0][Config.KEY_COLUMN].tolist()
mail_clients    = X_test_noenc_nonorm[X_test_noenc_nonorm["interaction_mail_churn_mention"] > 0][Config.KEY_COLUMN].tolist()
engaged_clients = X_test_noenc_nonorm[X_test_noenc_nonorm["client_toujours_engage"] == 1][Config.KEY_COLUMN].tolist()

# 5 — Build final model with clip rules
final_model = (
    FinalModel(ensemble_model, global_threshold=Config.GLOBAL_THRESHOLD)
    .add_clip(nps_clients,     value=1, label="NPS clip     (→1):")
    .add_clip(mail_clients,    value=1, label="mail clip    (→1):")
    .add_clip(engaged_clients, value=0, label="engaged clip  (→0):")
)



NameError: name 'S3_BUCKET_FINAL_MODELS' is not defined

In [ ]:

# ── 10. Final summary ─────────────────────────────────────────────────────
print(f"\n Churn rate final:           {results['churn_final'].mean()*100:.1f}%  ({results['churn_final'].sum()} / {len(results)} clients)")

# ── 11. Save CSV ──────────────────────────────────────────────────────────
output = results[[Config.KEY_COLUMN, "churn_final"]].rename(columns={Config.KEY_COLUMN: Config.KEY_COLUMN, "churn_final": "churn_pred"})
output_path = "churn_predictions.csv"
output.to_csv(output_path, index=False)
print(f"Final prediction saved: {output_path}  ({len(output)} rows)")

In [ ]:
#AT THE END 
RocCurveDisplay --> stack model, stack then clipped model --> each individuals models (superposed)

In [ ]:
#stack model : 0.7891784392438536 ROC-AUC
#final model,: 0.7891784392438536